In [ ]:
!pip install -q arxiv pymupdf 

In [ ]:
import arxiv
import fitz
import json
import os
import time

In [ ]:
client = arxiv.Client()

queries = [
    "large language models transformer",
    "retrieval augmented generation RAG",
    "fine-tuning attention mechanisms"
]

all_results = []
seen_ids = set()

for query in queries:
    search = arxiv.Search(query=query, max_results=34)
    results = list(client.results(search))
    for r in results:
        if r.entry_id not in seen_ids:
            seen_ids.add(r.entry_id)
            all_results.append(r)
    time.sleep(3)

print(f"Total unique papers found: {len(all_results)}")

In [ ]:
import requests

os.makedirs('/kaggle/working/papers', exist_ok=True)

downloaded = []
failed = []

headers = {'User-Agent': 'Mozilla/5.0 (compatible; ResearchLens/1.0)'}

for paper in all_results:
    paper_id = paper.entry_id.split('/')[-1]
    pdf_path = f'/kaggle/working/papers/{paper_id}.pdf'
    
    if os.path.exists(pdf_path):
        downloaded.append({'id': paper_id, 'path': pdf_path, 'title': paper.title})
        continue
    
    try:
        url = f'https://arxiv.org/pdf/{paper_id}'
        response = requests.get(url, headers=headers, timeout=30)
        
        if response.status_code == 200 and len(response.content) > 10000:
            with open(pdf_path, 'wb') as f:
                f.write(response.content)
            downloaded.append({'id': paper_id, 'path': pdf_path, 'title': paper.title})
            print(f"✓ {paper_id}")
        else:
            failed.append({'id': paper_id, 'error': f'status {response.status_code}'})
            print(f"✗ {paper_id} — status {response.status_code}")
        
        time.sleep(2)
        
    except Exception as e:
        failed.append({'id': paper_id, 'error': str(e)})
        print(f"✗ {paper_id} — {str(e)}")

print(f"\nDownloaded: {len(downloaded)}")
print(f"Failed: {len(failed)}")

In [ ]:
def parse_pdf(pdf_path, paper_id, title):
    doc = fitz.open(pdf_path)
    pages = []
    
    for page_num, page in enumerate(doc):
        text = page.get_text()
        if len(text.strip()) > 50:
            pages.append({
                'paper_id': paper_id,
                'title': title,
                'page_num': page_num,
                'text': text.strip()
            })
    
    doc.close()
    return pages

all_pages = []
parse_errors = []

for paper in downloaded:
    try:
        pages = parse_pdf(paper['path'], paper['id'], paper['title'])
        all_pages.extend(pages)
    except Exception as e:
        parse_errors.append({'id': paper['id'], 'error': str(e)})

print(f"Total pages parsed: {len(all_pages)}")
print(f"Parse errors: {len(parse_errors)}")
print(f"Sample page text (first 300 chars):")
print(all_pages[0]['text'][:300])

In [ ]:
with open('/kaggle/working/parsed_papers.json', 'w') as f:
    json.dump(all_pages, f)

print(f"Saved {len(all_pages)} pages to parsed_papers.json")

In [ ]:
metadata = []
for paper in all_results:
    paper_id = paper.entry_id.split('/')[-1]
    metadata.append({
        'id': paper_id,
        'title': paper.title,
        'authors': [a.name for a in paper.authors],
        'abstract': paper.summary,
        'published': str(paper.published),
        'url': paper.entry_id
    })

with open('/kaggle/working/papers_metadata.json', 'w') as f:
    json.dump(metadata, f)

print(f"Saved metadata for {len(metadata)} papers")

In [ ]:
!pip install -q "datasets==2.20.0"

In [ ]:
!pip show pyarrow | grep Version

In [ ]:
from datasets import load_dataset

In [ ]:
qasper = load_dataset("allenai/qasper")

print(f"QASPER splits: {qasper.keys()}")
print(f"Train size: {len(qasper['train'])}")
print(f"Validation size: {len(qasper['validation'])}")
print(f"Test size: {len(qasper['test'])}")
print(f"\nSample entry keys: {qasper['train'][0].keys()}")

In [ ]:
sample = qasper['train'][0]
print(f"Top level keys: {sample.keys()}")
print(f"\nqas type: {type(sample['qas'])}")
print(f"\nqas sample: {sample['qas']}")

In [ ]:
def extract_qa_pairs(dataset_split, max_pairs=200):
    pairs = []
    
    for entry in dataset_split:
        title = entry['title']
        questions = entry['qas']['question']
        answers_list = entry['qas']['answers']
        
        for question, answer_obj in zip(questions, answers_list):
            for answer in answer_obj['answer']:
                if answer['unanswerable']:
                    continue
                
                free_form = answer['free_form_answer']
                if free_form and len(free_form) > 10:
                    pairs.append({
                        'question': question,
                        'answer': free_form,
                        'paper_title': title
                    })
                    break
            
            if len(pairs) >= max_pairs:
                return pairs
    
    return pairs

qasper_pairs = extract_qa_pairs(qasper['train'], max_pairs=200)

print(f"Extracted {len(qasper_pairs)} QA pairs from QASPER")
print(f"\nSample pair:")
print(f"Q: {qasper_pairs[0]['question']}")
print(f"A: {qasper_pairs[0]['answer'][:200]}")

In [ ]:
with open('/kaggle/working/qasper_pairs.json', 'w') as f:
    json.dump(qasper_pairs, f)

print(f"Saved {len(qasper_pairs)} QASPER pairs to qasper_pairs.json")

In [ ]:
files_to_check = [
    '/kaggle/working/parsed_papers.json',
    '/kaggle/working/papers_metadata.json',
    '/kaggle/working/qasper_pairs.json',
]

for filepath in files_to_check:
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"✓ {filepath.split('/')[-1]} — {size_mb:.2f} MB")
    else:
        print(f"✗ MISSING: {filepath}")

print(f"\nTotal papers directory:")
pdf_count = len([f for f in os.listdir('/kaggle/working/papers') if f.endswith('.pdf')])
print(f"✓ {pdf_count} PDFs in /kaggle/working/papers/")